# 06 · Loop engineering

Oltre all'agente base esistono altri "loop" che lo circondano
([The Art of Loop Engineering](https://www.langchain.com/blog/the-art-of-loop-engineering)).
Ne vediamo tre, ognuno costruito da zero e commentato:
1. **verifica** con una rubrica (Loop 2);
2. **trigger a eventi** (Loop 3);
3. **hill climbing**: dai dati d'uso a una proposta di miglioramento (Loop 4).

## Setup (autonomo)

Ogni notebook è **indipendente**: non importa nulla dal progetto. Qui carichiamo la chiave
API dal file `.env` e creiamo un modello. Esegui le celle in ordine dall'alto verso il basso.

In [ ]:
# Carichiamo le variabili d'ambiente dal file `.env`.
# Lo cerchiamo nella cartella corrente e in quelle superiori, così il notebook
# funziona sia se avviato dalla radice del progetto sia dalla cartella `notebooks`.
import os
from pathlib import Path

from dotenv import load_dotenv


def trova_env() -> Path:
    for cartella in (Path.cwd(), *Path.cwd().resolve().parents):
        if (cartella / ".env").is_file():
            return cartella / ".env"
    raise FileNotFoundError("File .env non trovato: copia .env.example in .env e aggiungi la chiave.")


env_file = trova_env()
load_dotenv(env_file, override=False)          # carica le variabili senza sovrascrivere quelle già presenti
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY mancante nel file .env"
print("Ambiente caricato da:", env_file)

In [ ]:
# `ChatOpenAI` è il wrapper LangChain attorno al modello.
# Lo creiamo una volta e lo riusiamo in tutto il notebook.
from langchain_openai import ChatOpenAI

MODELLO = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")   # modello economico, va bene per imparare
model = ChatOpenAI(
    model=MODELLO,
    use_responses_api=True,   # API "responses" di OpenAI
    store=False,              # non conservare la conversazione sui server OpenAI
)
print("Modello pronto:", MODELLO)

## Loop 1 · L'agente base

Il punto di partenza: un agente qualsiasi. Sarà l'oggetto che gli altri loop "circondano".

In [ ]:
from langchain.agents import create_agent

agente = create_agent(model=model, tools=[], system_prompt="Rispondi in modo utile e conciso.")


def esegui(domanda: str) -> str:
    esito = agente.invoke({"messages": [{"role": "user", "content": domanda}]})
    return esito["messages"][-1].text

## Loop 2 · Verifica con una rubrica

Un secondo modello fa da **giudice**: dà un voto 0–1 a più criteri e un feedback.
La soglia (passa/non passa) la decidiamo noi in Python — deterministica, non dal modello.

Nota: con lo *structured output* di OpenAI lo schema deve essere **chiuso** (campi espliciti),
quindi elenchiamo i criteri come campi, non come dizionario libero.

In [ ]:
# Lo schema del giudizio: un voto per criterio + un feedback. Campi espliciti = schema chiuso.
from pydantic import BaseModel, Field


class Giudizio(BaseModel):
    completezza: float = Field(ge=0, le=1)
    chiarezza: float = Field(ge=0, le=1)
    feedback: str = ""

In [ ]:
# Il giudice è il modello con output strutturato sullo schema Giudizio.
giudice = model.with_structured_output(Giudizio)


def valuta(domanda: str, risposta: str, soglia: float = 0.7):
    g = giudice.invoke([
        {"role": "system", "content": "Valuta la RISPOSTA rispetto alla DOMANDA, 0-1 per criterio."},
        {"role": "user", "content": f"DOMANDA: {domanda}\nRISPOSTA: {risposta}"},
    ])
    voto = (g.completezza + g.chiarezza) / 2      # media dei criteri
    return voto >= soglia, voto, g.feedback        # passa?, voto, feedback

In [ ]:
# Ciclo di verifica: se non passa, rimandiamo indietro il feedback e riproviamo.
domanda = "Spiega cos'è un checkpoint in LangGraph."
risposta = esegui(domanda)
passa, voto, feedback = valuta(domanda, risposta)
print("voto:", round(voto, 2), "passa:", passa)

if not passa:
    risposta = esegui(f"{domanda}\nMigliora tenendo conto di questo feedback: {feedback}")
    print("--- risposta migliorata ---")
print(risposta)

## Loop 3 · Trigger a eventi

Finora l'agente parte perché lo chiamiamo noi. Un **trigger** lo avvia al verificarsi di un
evento: un orario (cron) o un webhook. Costruiamo un mini valutatore di espressioni cron.

In [ ]:
from datetime import datetime, timezone


def campo_cron(campo: str, valore: int) -> bool:
    # Supporta '*', liste '1,2', e step '*/5'. Sufficiente per capire l'idea.
    if campo == "*":
        return True
    if campo.startswith("*/"):
        return valore % int(campo[2:]) == 0
    return valore in {int(x) for x in campo.split(",")}


def cron_combacia(espressione: str, momento: datetime) -> bool:
    minuto, ora = espressione.split()[:2]     # usiamo i primi due campi: minuto e ora
    return campo_cron(minuto, momento.minute) and campo_cron(ora, momento.hour)

In [ ]:
# Uno "scheduler" minimale: se l'espressione combacia con l'ora attuale, avvia l'agente.
adesso = datetime.now(timezone.utc)
if cron_combacia("* * * * *", adesso):        # '* * * * *' = ogni minuto -> combacia sempre
    print(esegui("Scrivi una frase che conferma l'avvio automatico."))

## Loop 4 · Hill climbing

L'idea più avanzata: usare i **dati d'uso** per migliorare la configurazione dell'agente.
Un agente d'analisi legge un piccolo report e propone modifiche, entro una lista sicura di
campi. **Propose-only**: la proposta va poi rivista da un umano prima di applicarla.

In [ ]:
# La proposta ha SOLO campi consentiti (whitelist strutturale) -> schema chiuso, override sicuri.
class Proposta(BaseModel):
    sintesi: str = ""
    aggiunta_al_prompt: str | None = None       # testo da aggiungere al system prompt
    max_chiamate_tool: int | None = None        # nuovo limite di tool call

In [ ]:
analista = model.with_structured_output(Proposta)

# Un finto report dei run recenti (in un sistema vero verrebbe dai log/trace).
report = "Run totali: 8\nRun falliti: 3\nErrore ricorrente: retry ripetuti su web_read"

proposta = analista.invoke([
    {"role": "system", "content": "Analizza il REPORT e proponi migliorie solo se giustificate."},
    {"role": "user", "content": report},
])
print("Sintesi:", proposta.sintesi)
print("Aggiunta al prompt:", proposta.aggiunta_al_prompt)
print("Nuovo limite tool:", proposta.max_chiamate_tool)

## Prova tu

- Nel Loop 2, abbassa la soglia a 0.9 e osserva quante iterazioni servono.
- Nel Loop 4, applica davvero la proposta ricreando l'agente con `system_prompt` aggiornato —
  ma solo dopo averla letta: è il principio *propose-only + revisione umana*.

**Idea chiave**: l'agente è il cuore; verifica, eventi e auto-miglioramento sono i loop che
lo rendono affidabile, autonomo e capace di migliorare nel tempo.